In [1]:
from scipy.io import loadmat    # used to turn .mat files into numpy Arrays
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
def plot_ABP_PPG_signal(ABP, PPG, fs):

    t = np.arange(len(PPG)) / fs    # time axis, in seconds 
    
    plt.figure(figsize=(12,6))

    plt.subplot(2,1,1)
    plt.plot(t, ABP, color='blue')
    plt.title('ABP Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Normalized ABP ([0,1])')

    plt.subplot(2,1,2)
    plt.plot(t, PPG, color='red')
    plt.title('PPG Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Light Intensity [V]')

    plt.tight_layout()

    plt.show()

In [3]:
def unwrap_matlab_array(x):
    '''
    Recursively unwrap MATLAB object arrays to a plain Python number
    '''
    x = np.asarray(x)
    while x.dtype == object:
        x = x[0]
    if np.size(x) == 1:
        return x.item()
    return x


In [ ]:
data_path = 'ABP_PPG'
data_file = sorted(os.listdir(data_path))
rows = []
print(len(data_file))
for i in range(1000):
    file_path = os.path.join(data_path, data_file[i])
    data = loadmat(file_path)
    print(f'accessing data point {i}')
    for j in range(1):
                
        sp = data['signal_processing'][0,j]
        segment = sp['signal']

        ABP_signal = segment[0]
        PPG_signal = segment[1]

        beats_ppg = sp['struct_features']['ppg'][0,0]
        beats_abp = sp['struct_features']['beats_abp'][0,0]
        beats_masked = np.where(beats_abp == 0, np.nan, beats_abp)  # Replace 0 padding with NaN
        #print(np.nanmin(beats_masked[0]))

        #print(beats_ppg.dtype.names)
        n_beats = beats_ppg.shape[1]
        #print(n_beats)
        #print(beats_abp.shape[0])
        
        for beat_idx in range(0,n_beats):
            
            # beats_ppg some times has 1 more beat than beats_abp,
            if beat_idx >= beats_abp.shape[0]:
                continue

            SBP_val = np.nanmax(beats_masked[beat_idx])
            DBP_val = np.nanmin(beats_masked[beat_idx])

            row = {
                'data_ID': data_file[i],
                'struct': j,
                'beat_idx': beat_idx,
                'SBP': SBP_val,
                'DBP': DBP_val
            }

            feature_values = []
            beat_features = beats_ppg[0,beat_idx]   # contains all features for the current heartbeat beat_idx

            for num_feat in range(0,len(beats_ppg[0,0])):
                
                feature_location = int(unwrap_matlab_array(beat_features[num_feat]))
                
                if feature_location < 0 or feature_location >= len(PPG_signal):
                    continue

                feature_values.append(PPG_signal[feature_location])
                feature_name = beats_ppg.dtype.names[num_feat]
                row[f'{feature_name}'] = PPG_signal[feature_location]

            rows.append(row)
            # print(rows)

        df = pd.DataFrame(rows)
        df.to_csv('PPG_features_per_heartbeat.csv', index=False)


5857
accessing data point 0
1876
accessing data point 1
1876
accessing data point 2
1876
accessing data point 3
1876
accessing data point 4
1876
accessing data point 5
1876
accessing data point 6
1876
accessing data point 7
1876
accessing data point 8
1876
accessing data point 9
1876
accessing data point 10
1876
accessing data point 11
1876
accessing data point 12
1876
accessing data point 13
1876
accessing data point 14
1876
accessing data point 15
1876
accessing data point 16
1876
accessing data point 17
1876
accessing data point 18
1876
accessing data point 19
1876
accessing data point 20
1876
accessing data point 21
1876
accessing data point 22
1876
accessing data point 23
1876
accessing data point 24
1876
accessing data point 25
1876
accessing data point 26
1876
accessing data point 27
1876
accessing data point 28
1876
accessing data point 29
1876
accessing data point 30
1876
accessing data point 31
1876
accessing data point 32
1876
accessing data point 33
1876
accessing data poin

KeyboardInterrupt: 